# FolkIn — Fine-tuning model deteksi HP (YOLO)

Notebook ini melatih model object detection pretrained. Dataset harus diunggah melalui tombol Files saat sel upload dijalankan.

**Catatan:** model ini mendeteksi kelas `phone_in_hand` dan `phone`, bukan niat atau distraksi secara pasti. Periksa hasil prediksi dan evaluasi sebelum memakai alarm otomatis.


In [ ]:
!pip -q install -U ultralytics


## 1. Unggah dataset hasil konversi

Unggah `FolkIn_Phone_Detection_Dataset_YOLO.zip` yang disediakan bersama notebook ini.


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
import os, zipfile

zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if zip_name is None:
    raise FileNotFoundError("Unggah file ZIP dataset FolkIn terlebih dahulu.")

DATA_DIR = "/content/folkin_dataset"
os.makedirs(DATA_DIR, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(DATA_DIR)

DATA_YAML = os.path.join(DATA_DIR, "data.yaml")
print("Dataset config:", DATA_YAML)
print("Isi folder:", os.listdir(DATA_DIR))


## 2. Pastikan GPU aktif

Di Colab pilih **Runtime → Change runtime type → T4 GPU** jika tersedia.


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Fine-tuning YOLO26n

Mulai dengan 30 epoch sebagai percobaan awal. Jika hasil belum stabil, periksa anotasi dan tambah data yang relevan sebelum hanya menaikkan epoch.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")  # pretrained weights; bukan melatih model dari nol
results = model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=512,
    batch=8,
    patience=8,
    project="/content/folkin_runs",
    name="phone_detection",
    seed=42,
    plots=True
)


## 4. Evaluasi pada test set

Jangan menilai model hanya dari contoh training. Lihat precision, recall, mAP50, dan mAP50-95.


In [ ]:
best_path = "/content/folkin_runs/phone_detection/weights/best.pt"
best_model = YOLO(best_path)
metrics = best_model.val(data=DATA_YAML, split="test", imgsz=512)
print(metrics)


## 5. Uji pada beberapa gambar test

Periksa apakah kotak benar-benar mengelilingi area HP yang relevan. Karena anotasi kelas `phone_in_hand` sumber awalnya berupa poligon sekitar tangan dan HP, bounding box kelas ini dapat mencakup tangan.


In [ ]:
test_images = os.path.join(DATA_DIR, "test", "images")
sample_images = [
    os.path.join(test_images, f)
    for f in os.listdir(test_images)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
][:12]

predictions = best_model.predict(
    source=sample_images,
    conf=0.35,
    imgsz=512,
    save=True,
    project="/content/folkin_predictions",
    name="test_samples"
)
print("Hasil tersimpan di /content/folkin_predictions/test_samples")


## 6. Unduh bobot terbaik

File `best.pt` dipakai oleh script webcam lokal.


In [ ]:
from google.colab import files
files.download(best_path)


## 7. Catatan eksperimen untuk laporan

Catat versi dataset, jumlah gambar pada setiap split, epoch, ukuran gambar, batch size, precision, recall, mAP50, mAP50-95, serta contoh false positive dan false negative. Dataset sumber berlisensi CC BY 4.0; cantumkan atribusi kepada sumber Roboflow Universe dalam laporan.
